# # 1. 操作系统概述

## ## 1.1 操作系统概述

### ### 1.1.1 准备一些函数

In [3]:
import os
import ipywidgets as widgets
import IPython.display as display1
import pygments
from pygments.lexers import get_lexer_by_name
from pygments.formatters import HtmlFormatter

def display_folder_contents(folder_path):
    def highlight_code(code, lexer_name):
        formatter = HtmlFormatter(style='colorful')
        highlighted_code = pygments.highlight(code, get_lexer_by_name(lexer_name), formatter)
        css = "<style>{}</style>".format(formatter.get_style_defs('.highlight'))
        return css + highlighted_code

    # Mapping of file extensions to Pygments lexer names
    lexer_mapping = {
        '.py': 'python',
        '.cpp': 'cpp',
        '.c': 'c',
        '.md': 'markdown',
        '.Makefile': 'makefile',
        '.txt': 'text',
        '.S': 'asm',
        '.asm': 'asm'
    }

    # Get a list of all files in the folder
    file_names = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    # Sort list of all files, support only 0-9
    file_names.sort(key=lambda x:int(x[:1]))
    
    # Create a tab for each file
    tabs = widgets.Tab()
    tab_contents = []
    for file_name in file_names:
        # Read the contents of the file
        with open(os.path.join(folder_path, file_name), 'r') as f:
            code = f.read()
        # Determine the lexer name based on the file extension
        lexer_name = lexer_mapping.get(os.path.splitext(file_name)[1], 'text')
        # Add a tab with the highlighted code
        tab_contents.append(widgets.HTML(value=highlight_code(code, lexer_name)))
    tabs.children = tab_contents
    for i, file_name in enumerate(file_names):
        tabs.set_title(i, file_name[2:])

    display1.display(tabs)


In [4]:
from IPython.display import display, IFrame

def display_html_file(file_path, width=550, height=350):
    """
    Displays an HTML file in a Jupyter notebook using an inline frame.
    Arguments:
        - file_path (str): Path to the HTML file.
        - width (int): Width of the inline frame in pixels (default: 550).
        - height (int): Height of the inline frame in pixels (default: 350).
    """
    display(IFrame(file_path, width=width, height=height))

背景回顾：大家已经有多年的操作系统使用经验，也知道 “操作系统” 是一门很重要的课程，但却无法回答 “操作系统到底是什么”、“为什么重要” 这样的问题。我们将在一学期的时间里回答这些问题，并建立操作系统相关的基本概念、思想方法和编程技巧。

本讲内容：课程简介、操作系统的发展历史，回答三个问题：

- (Why): 为什么要学操作系统？
- (What): 到底什么是操作系统？
- (How): 怎么学操作系统？

In [5]:
display_html_file("slides/1.2.html")

In [8]:
display_html_file("slides/1.3.html")

### ### 1.1.2 Demo：模拟数字系统

In [7]:
display_folder_contents("logisim")

logisim.c 代码的输出作为 seven-seg.py 代码的输入。

$ ./a.out | python3 seven-seg.py

### ### 1.1.3 Demo：模拟RISC-V指令执行

In [8]:
display_folder_contents("rvemu")

In [9]:
display_html_file("slides/1.4.html")

## ## 1.2 Take-away Messages

操作系统没有传说中那么复杂 (程序视角：对象 + API，硬件视角：一个 C 程序)

- 为什么要学操作系统：解锁 “实现一切” 的系统编程能力
- 什么是操作系统：应用视角 (一组对象 + API)、机器视角 (一个程序)
- 怎么学操作系统：答案就在代码中

### ### 1.2.1 延伸阅读

#### #### 阅读材料

教科书 [Operatin Systems: Three Easy Pieces:](https://pages.cs.wisc.edu/~remzi/OSTEP/)

- Preface 前言
- 第1章 - Dialogue on the Book
- 第2章 - Introduction to Operating Systems

延伸阅读：

- [Harley Hahn's Guide to Unix and Linux](http://www.harley.com/books/sg3.html)

#### #### 熟悉基础命令行工具
   
熟悉一些典型的 Linux 的命令行工具：

- 查找文档的工具，例如 tldr 和 man
- UNIX 系统管理的基础工具，请 STFW
- 在命令行使用 gcc 编译、运行一个 C 程序，阅读 gcc --help 的帮助信息了解 gcc 支持的主要功能
- GNU Make - 本课程使用 GNU Make 构建项目。我们避免了 CMake 这些在实际项目中很常用，但会导致非常冗长 (也难以理解) 流程的工具
- Git - 本课程所有代码都用 Git 管理；虽然严格意义说你不需要知道 Git 是如何工作的，但学会使用它会对你今后的职业生涯很有帮助


- [参考资料](https://jyywiki.cn/OS/OS_References)
- [The Art of Command Line](https://github.com/jlevy/the-art-of-command-line)
- [Modern Unix](https://github.com/ibraheemdev/modern-unix)

# # 2. 应用视角的操作系统

**背景回顾**：操作系统有三条主线：“软件 (应用)”、“硬件 (计算机)”、“操作系统 (软件直接访问硬件带来麻烦太多而引入的中间件)”。想要理解操作系统，对操作系统的服务对象 (应用程序) 有精确的理解是必不可少的。

**本讲内容**：指令序列和高级语言的状态机模型；回答以下问题：

- 什么是软件 (程序)？
- 如何在操作系统上构造最小/一般/图形界面应用程序？
- 什么是编译器？编译器把一段程序翻译成什么样的指令序列才算 “正确”？

In [71]:
display_html_file("slides/2.1.html")

In [64]:
display_folder_contents("minimal")

对于任意的 C 语言代码，我们都可以把它解析成语法树的结构 (类似于表达式树，在《计算机系统基础》的 Programming Assignment 中包含了类似的实验)。C 程序的语义解释执行 “一条语句” 的更严谨说法是解释执行当前语句中 “优先级最高的节点”。

此外，我们也可以用类似汉诺塔的方法把 C 语言改写成仅包含顺序执行的简单语句和条件跳转 if (cond) goto 的 “简化 C 语言代码”——循环、分支和函数调用都被消灭了。这时候，我们的 C 代码已经可以逐条翻译成汇编指令了。这也是为什么 C 语言被称为 “高级的汇编语言”——我们几乎总是能从 C 语言的语法层面对应到机器指令序列。

In [65]:
display_html_file("slides/2.3.html")

## ## 2.1 Demo: 在代码中控制编译优化

In [66]:
display_folder_contents("compiler-opt")

我们甚至有可能 “证明” 编译器所作出的优化全部是正确的，或是用更激进地方式生成更优化的代码，例如把几个行为已知的系统调用 “合并” 起来。如果你对这方面的知识感兴趣，可以参考原文中列出的论文。

In [10]:
display_html_file("slides/2.4.html")

tip: 从 busybox、toybox 上学习 vi

## ## 2.2 Demo：使用 strace 跟踪系统调用

In [76]:
display_folder_contents("strace")

1. 预处理，在此阶段，编译器会将代码中的头文件 <stdio.h> 等编译进来；

    ```
    gcc -E test.c -o test.i
    ```

2. 编译，gcc 把代码翻译为汇编语言

    ```
    gcc -S test.i -o test.s
    ```

3. 汇编，gcc 把汇编代码编译为机器代码

    ```
    gcc -c test.s -o test.o
    ```

4. 链接

    ```
    gcc test.o -o test
    ```

    > 在源程序中没有 printf 等函数的实现，于是前往步骤 2 时包含进来的 <stdio.h> 文件，发现其中有 printf 函数的声明，但没有函数的实现；
    >
    > Linux 系统把 printf 函数的实现都放在了 libc.so.6 的库文件中，在没有参数指定时，g++ 到系统默认的路径 /usr/lib 下查找库文件，将函数链接到 libc.so.6 库函数中去，这样就有了 printf 函数的实现部分，把程序中一些函数的实现部分链接起来，这就是链接阶段的工作；

```bash
# 我们的 minimal.S 直接是汇编语言，所以省略了前两步
$ gcc -c minimal.S     # 编译汇编文件
$ objdump -d minimal.o # 使用 objdump 工具查看汇编代码
$ ld minimal.o         # 也可以使用 ld 命令链接
$ ./a.out              # 输出 hello, os wolrd
$ objdump -d a.out | less # 还可以用 objdump 查看它，是一串非常短的代码，事实上这也是最小的 hello, world
```

-----

继续追问，我们为什么会想到使用 `SYS_write`、`SYS_exit` 这些变量，它们是在哪里被定义的？

```bash
$ man 2 syscall
$ man 2 syscalls  # 完整的系统调用的列表
```

----- 

进入这部分的主题：**strace**

```bash
strace ./a.out    # 追踪 minimal.S 这个程序所有的系统调用
```

如果我们想知道 gcc 的系统调用呢？

```bash
$ strace -f gcc hello.c
$ strace -f gcc hello.c |& vim -
```

```
:%!grep -e execve
:%!grep -v ENOENT
```


> execve("/usr/bin/gcc", ["gcc", "hello.c"], [/*...... 
>
> execve("/usr/lib/gcc/x86_64-linux-gnu/6/cc1", ...... 
>
> execve("/usr/bin/as", ["as", "--64", "-o", "/t...... 
>
> execve("/usr/lib/gcc/x86_64-linux-gnu/6/collec...... 
>
> execve("/usr/bin/ld", ["/usr/bin/ld", "-plugin...... 
```

## ## 2.3 Take-away Messages

无论是汇编代码还是高级语言程序，它们都可以表示成状态机：

- 高级语言代码 .c：状态：栈、全局变量；状态迁移：语句执行
- 汇编指令序列 .s：状态：(M, R)；状态迁移：指令执行
- 编译器实现了两种状态机之间的翻译

应用程序与操作系统沟通的唯一桥梁是系统调用指令 (例如 x86-64 的 syscall)。计算机系统不存在玄学；一切都建立在确定的机制上

### ### 2.3.1 课后习题/编程作业

#### #### 阅读材料

- 浏览 [GNU Coreutils](https://www.gnu.org/software/coreutils/) 和 [GNU Binutils](https://www.gnu.org/software/binutils/) 的网站，建立 “手边有哪些可用的命令行工具” 的一些印象。
- 浏览 [gdb](https://sourceware.org/gdb/current/onlinedocs/gdb.html/) 文档的目录，找到你感兴趣的章节了解，例如——“Reverse Execution”、“TUI: GDB Text User Interface”……
- 对于你有兴趣的命令行工具，可以参考 [busybox](https://www.busybox.net/) 和 [toybox](https://landley.net/toybox/about.html) 项目中与之对应的简化实现。Toybox 现在已经成为了 Android 自带的命令行工具集。

#### #### 编程实践

在你的 Linux 中运行课堂上的代码示例，包括：

- 编译链接最小二进制文件，并使用 strace 查看执行的系统调用序列。
- 使用 strace 查看更多程序的系统调用序列，并在好奇心的驱使下了解一部分系统调用的作用。可以借助互联网、Stackoverflow 或是 ChatGPT (但在使用时，要小心求证结果)。